# The two datasets, at two scales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/datasets.ipynb)

Built from [`cookbook/book/chapters/datasets.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/datasets.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

The book runs on two public graphs, each used where it is strongest.

- **Air Routes** — Neptune's own teaching dataset: 3,504 airports, a `route`
  graph (airport↔airport) and a `contains` hierarchy
  (continent→country→airport). Numeric and categorical features, thin text.
  It carries the **tiers 01–02 on-ramp**.
- **ogbn-arxiv** — ~169k computer-science papers, ~1.17M citation edges, 40
  subject classes, title + abstract. It carries the **tiers 03–04 spine**, and
  is the dataset the graph-conformal literature reports on.

## Two scales, one book

Every chapter runs its capability live. What it runs over is a **scale**:

- **`small`** — the committed fixtures and the tiny fixture encoders, on the
  CPU: seconds per chapter. This is what the book's CI renders, and what a
  CPU Colab runtime runs.
- **`full`** — the published datasets and real encoders (ModernBERT for text),
  on a GPU: the book's findings at the size they are about.

The chapter code is the same at both; the data, the encoders and the frozen
goldens a run is checked against differ. Choose with the environment variable
`JAMMI_COOKBOOK_SCALE` (`small` when unset) before the chapter starts:

In [ ]:
from jammi_cookbook import scale

SCALE = scale.current()
print(f"this run: scale = {SCALE}")

Air Routes is small enough to run whole, so both scales read the same graph
(committed with the cookbook). ogbn-arxiv at `full` is a **connected ball** of
the citation graph — 4,000 papers collected breadth-first from the most-cited
paper, downloaded and checksum-verified on first use; a ball keeps the citation
density and the subject homophily later chapters depend on. At `small` it is
that ball's 400 best-connected papers (committed with the cookbook): one
dataset at two sizes, the smaller keeping the larger's character.

In [ ]:
import tempfile

import jammi
from jammi_cookbook import contracts, datasets

db = jammi.connect(f"file://{tempfile.mkdtemp()}")
air = datasets.air_routes(db)
arxiv = datasets.arxiv(db, SCALE)

A file source is queried in SQL as `<source>.public.<table>`, where the table
is the file's name without its extension. The book names each file after its
source, so every source here reads as `<source>.public.<source>`.

## Air Routes

In [ ]:
def count(source: str) -> int:
    return db.sql(f"SELECT COUNT(*) AS n FROM {source}.public.{source}").to_pylist()[0]["n"]


print(f"airports:       {count(air.airports)}")
print(f"route edges:    {count(air.routes)}")
print(f"contains edges: {count(air.contains)}")

# An airport's continent comes from the continent→airport hierarchy.
atl = db.sql(
    f"SELECT code, city, country, continent FROM {air.airports}.public.{air.airports} "
    "WHERE code = 'ATL'"
).to_pylist()[0]
print("ATL:", atl)

In [ ]:
contracts.assert_close("datasets.air_airports", count(air.airports))
contracts.assert_close("datasets.air_routes", count(air.routes))
contracts.assert_close("datasets.air_contains", count(air.contains))
assert atl["continent"] == "NA"

## ogbn-arxiv

The time split is the dataset's own: papers up to 2017 train, 2018 validates,
2019 onward is the test era — a split by publication year.

In [ ]:
print(f"papers:          {count(arxiv.papers)}")
print(f"citation edges:  {count(arxiv.cites)}")
print("time split:     ", {name: len(ids) for name, ids in arxiv.split.items()})

top = db.sql(
    f"SELECT subject, COUNT(*) AS c FROM {arxiv.papers}.public.{arxiv.papers} "
    "GROUP BY subject ORDER BY c DESC LIMIT 3"
).to_pylist()
print("top subjects:   ", [(r["subject"], r["c"]) for r in top])

The citation graph is **homophilous** on the subject label: a paper cites
papers of its own subject more often than the subject mix alone would give —
the chance level is the probability that two papers drawn at random share a
subject, $\sum_s p_s^2$. That correlation, together with the time split, is
what later breaks conformal exchangeability (tier 04).

In [ ]:
edges = db.sql(
    f"SELECT a.subject AS sa, b.subject AS sb "
    f"FROM {arxiv.cites}.public.{arxiv.cites} e "
    f"JOIN {arxiv.papers}.public.{arxiv.papers} a ON e.src = a.paper_id "
    f"JOIN {arxiv.papers}.public.{arxiv.papers} b ON e.dst = b.paper_id"
).to_pylist()
homophily = sum(r["sa"] == r["sb"] for r in edges) / len(edges)
mix = db.sql(
    f"SELECT COUNT(*) AS c FROM {arxiv.papers}.public.{arxiv.papers} GROUP BY subject"
).column("c").to_pylist()
chance = sum((c / sum(mix)) ** 2 for c in mix)
print(f"citation-edge subject homophily: {homophily:.3f}  (chance for this subject mix: {chance:.3f})")

In [ ]:
contracts.assert_close("datasets.arxiv_papers", count(arxiv.papers))
contracts.assert_close("datasets.arxiv_cites", count(arxiv.cites))
contracts.assert_close("datasets.arxiv_homophily", homophily, tol=1e-9)
assert homophily > chance

An embedded engine holds its catalog until `close()` returns, so a chapter
closes its session before anything removes the directory the catalog lives in.

In [ ]:
db.close()